<a href="https://colab.research.google.com/github/xmayksx1/DAX-BI/blob/main/Factor_Semanal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# ==========================================
# --- DEFINICIÓN DE CLASES Y CARGA DE DATOS ---
# ==========================================
class DatosVenta:
    def __init__(self):
        self.ruta_carpeta = r"C:\Users\Usuario\OneDrive - Aceitera El Real\Escritorio\Planificación - Planificación y Control\35. Dash Temporal 2026\Histórico CSV Venta"
        self.archivos = glob.glob(os.path.join(self.ruta_carpeta, "*.csv"))

        if not self.archivos:
            print("No se encontraron archivos CSV en la ruta especificada.")
            self.df_total = pd.DataFrame()
        else:
            lista_df = [pd.read_csv(f, encoding='latin1', sep=None, engine='python') for f in self.archivos]
            self.df_total = pd.concat(lista_df, ignore_index=True)

class Dimensional_Productos:
    def __init__(self):
        self.ruta = r"C:\Users\Usuario\OneDrive - Aceitera El Real\Escritorio\Planificación - Planificación y Control\35. Dash Temporal 2026\Modelos de Prediccion\Tabla Dimensional de Proveedor_Sublinea_Linea.xlsx"
        self.df_dim = pd.read_excel(self.ruta)

# Carga de datos
datos = DatosVenta()
dim_prod = Dimensional_Productos()

# Join de tablas
venta_df = pd.merge(
    datos.df_total,
    dim_prod.df_dim,
    left_on='ITMCodigo',
    right_on='Codigo',
    how='inner'
)

# ==========================================
# --- TRANSFORMACIÓN DE FECHAS CORREGIDA ---
# ==========================================
columna_fecha = 'FECHAFACTURA'

# 1. Parseo estricto Día/Mes/Año (dd/mm/yyyy)
venta_df[columna_fecha] = pd.to_datetime(venta_df[columna_fecha], format='%d/%m/%Y', errors='coerce')

# Filtramos filas que no sean fechas válidas
venta_df = venta_df.dropna(subset=[columna_fecha])

# 2. Creación de columnas con formato de tiempo
# %Y-%U: Semana del año iniciando en DOMINGO (00 a 53)
venta_df['FASEMANA'] = venta_df[columna_fecha].dt.strftime('%Y-%U')
venta_df['FANOMES'] = venta_df[columna_fecha].dt.strftime('%Y%m')   # Sin guion (ej. 202604)
venta_df['FANO'] = venta_df[columna_fecha].dt.strftime('%Y')        # Solo año (ej. 2026)
venta_df['FMES'] = venta_df[columna_fecha].dt.strftime('%m')        # Solo mes (ej. 04)

# ==========================================
# --- AGRUPACIÓN Y RELLENO DE CEROS ---
# ==========================================

# Definir las columnas
cols_tiempo = ["FASEMANA", "FANOMES", "FANO", "FMES"]
cols_categoricas = ["ITMCodigo", "ITMDescripcion", "Bonificado", "ZGDepartamento", "Proveedor", "Sublinea"]

# Agrupación inicial (solo ventas reales)
venta_df_agrupado = venta_df.groupby(cols_tiempo + cols_categoricas).sum(numeric_only=True).reset_index()

# A. Extraer combinaciones únicas de semanas
df_tiempo_unico = venta_df[cols_tiempo].drop_duplicates()

# B. Extraer combinaciones únicas de productos/categorías
df_categorico_unico = venta_df[cols_categoricas].drop_duplicates()

# C. Crear la matriz cruzada completa (Semanas x Productos)
base_completa = df_tiempo_unico.merge(df_categorico_unico, how='cross')

# D. Unir la matriz completa con la venta real
venta_df_final = pd.merge(
    base_completa,
    venta_df_agrupado,
    on=cols_tiempo + cols_categoricas,
    how='left'
)

# E. Rellenar con 0 las semanas que no tuvieron venta
columnas_numericas = venta_df_final.select_dtypes(include=[np.number]).columns
metricas = [col for col in columnas_numericas if col not in cols_tiempo + cols_categoricas]
venta_df_final[metricas] = venta_df_final[metricas].fillna(0)

# ==========================================
# --- EXPORTACIÓN ---
# ==========================================
ruta_salida = r"C:\Users\Usuario\OneDrive - Aceitera El Real\Escritorio\Planificación - Planificación y Control\35. Dash Temporal 2026\Modelos de Prediccion\Resultado_Ventas_Agrupado_Semanal.csv"
venta_df_final.to_csv(ruta_salida, index=False, encoding='utf-8-sig', sep=';')

print(f"Archivo guardado exitosamente en: {ruta_salida}")

# Si ejecutas esto en Jupyter Notebook, display() funcionará perfectamente.
# Si es un script puro de Python (.py), puedes cambiarlo por print(venta_df_final.head())
try:
    display(venta_df_final)
except NameError:
    print(venta_df_final.head())